In [10]:
import torch
import transformers as tf
import plotly as pt
import numpy as np
import sklearn as ml

In [11]:
model_name  = "google/gemma-3-1b-pt" 
cfg = tf.AutoConfig.from_pretrained(model_name)
model = tf.AutoModelForCausalLM.from_pretrained(
    model_name,
    config=cfg,
    attn_implementation="eager",
    torch_dtype=torch.float32,
)
model.to("cpu")
tokenizer = tf.AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [12]:
config = {}
config["LAYERS"] = getattr(cfg, 'num_hidden_layers', 'N/A')
config["HIDDEN_SIZE"] = getattr(cfg, 'hidden_size', 'N/A')
config["ATTENTION_HEADS"] = getattr(cfg, 'num_attention_heads', 'N/A')
config["KV_HEADS"] = getattr(cfg, 'num_key_value_heads', 'N/A')
config["SLIDING_WINDOW"] = getattr(cfg, 'sliding_window', 'N/A')
config['FFN'] = getattr(cfg, 'intermediate_size', 'N/A')
config['VOCAB_SIZE'] = getattr(cfg, 'vocab_size', 'N/A')
config['FINAL_LOGIT_SOFTCAP'] = getattr(cfg, 'final_logit_softcapping', 'N/A')
config['ATTN_LOGIT_SOFTCAP'] = getattr(cfg, 'attn_logit_softcapping', 'N/A')
for name, module in model.named_modules():
    config[f"{name}"] = type(module).__name__

In [13]:
prompt = """Capital of
France is"""
inputs = tokenizer(prompt, return_tensors="pt")
print(inputs)
print(" ".join([tokenizer.convert_ids_to_tokens(i) for i in inputs.input_ids[0].tolist()]))

{'input_ids': tensor([[    2, 64753,   529,   107, 31756,   563]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
<bos> Capital ▁of 
 France ▁is


In [38]:
def token_label_helper(id_):
    label = tokenizer.convert_ids_to_tokens(id_)
    label = label.replace("▁", "·")
    label = label.replace("\n", "⏎")
    label = label.replace("\t", "⇥")
    return label
def prompt_token_labels(inputs_tnsr):
    # inputs = inputs_tnsr[0].tolist()
    labels = [token_label_helper(i) for i in inputs_tnsr]
    if len(labels) != len(inputs_tnsr):
        raise AssertionError("Length of labels and tokens do not match")
    return ([label if len(label) < 10 else label[:10] + "..." for label in labels],labels)
    

In [39]:
prompt_token_labels(inputs.input_ids[0].tolist())

(['<bos>', 'Capital', '·of', '·Ethiopia', '·is'],
 ['<bos>', 'Capital', '·of', '·Ethiopia', '·is'])

# Appendix : Tests

In [28]:
prompt = "Capital of Ethiopia is"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Capital of Ethiopia is the name given by the English to the capital city of the Ethiopian nation. Its name comes from the capital city of the Ethiopian Empire and was the official name of the capital city for over one century. The new name was given to the capital in 


In [40]:
from test_token_labels import check_token_labels
check_token_labels(tokenizer, token_label_helper, prompt_token_labels)



-- fact: 'The capital of France is'
pos       id  raw piece               full label              short label
  0        2  '<bos>'                 <bos>                   <bos>
  1      818  'The'                   The                     The
  2     5279  '▁capital'              ·capital                ·capital
  3      529  '▁of'                   ·of                     ·of
  4     7001  '▁France'               ·France                 ·France
  5      563  '▁is'                   ·is                     ·is

-- code: 'def my_func(x):\n    y = obj.__init__(x)\n\treturn y_value'
pos       id  raw piece               full label              short label
  0        2  '<bos>'                 <bos>                   <bos>
  1     2063  'def'                   def                     def
  2     1041  '▁my'                   ·my                     ·my
  3   236779  '_'                     _                       _
  4     6823  'func'                  func                    func
  5   

False

In [18]:
# import torch

# # 1. Is Metal available?
# print(torch.backends.mps.is_available())  # True/False for Metal

# # 2. Is it built into PyTorch?
# print(torch.backends.mps.is_built())  # True/False

# # 3. Check detailed info
# print(f"MPS available: {torch.backends.mps.is_available()}")
# print(f"MPS built: {torch.backends.mps.is_built()}")

# # 4. Move tensor to Metal GPU
# if torch.backends.mps.is_available():
#     x = torch.randn(10, 10).to("mps")
#     print(x.device)  # torch.device('mps:0')
# else:
#     print("Metal GPU not available, falling back to CPU")

# # 5. Check what device you're on
# device = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {device}")